# AIE S4 — Warm-up 2: CPU vs GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/racousin/data_science_practice/blob/main/website/public/modules/python-ai-engineering/challenges/aie-s4-cpu-gpu-benchmark.ipynb)

**No API key needed.** Ten minutes, and it answers a question you
are about to have: *do I need a GPU for this course?*

The honest answer is no, and this notebook is how you find that out
rather than being told. It measures three things on whatever machine
you are running on:

1. a large matrix multiplication — the operation a neural network is
   mostly made of
2. training the MLP from warm-up 1, scaled up
3. **the crossover** — the model size below which the GPU is
   *slower*, because moving the data costs more than the arithmetic
   saves

> **Enable the GPU first.** In Colab: **Runtime → Change runtime type
> → Hardware accelerator → GPU**, then **Save**, then run all. Without
> one every cell still runs and reports CPU timings only.

---

## 0. What are we running on?

In [ ]:
import time

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

HAS_GPU = torch.cuda.is_available()
print(f"torch          : {torch.__version__}")
print(f"CPU threads    : {torch.get_num_threads()}")
print(f"CUDA available : {HAS_GPU}")
if HAS_GPU:
    print(f"GPU            : {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU memory     : {props.total_memory / 1e9:.1f} GB")
else:
    print()
    print("No GPU. Runtime > Change runtime type > GPU, then Run all.")
    print("Everything below still runs; the GPU columns will say n/a.")

---

## 1. Matrix multiplication

Start with the primitive. A layer's forward pass **is** a matrix
multiply: `nn.Linear(in, out)` computes `X @ W.T + b`. Whatever the
hardware does to a matmul, it does to your network.

Multiplying two n × n matrices costs about $2n^3$ floating-point
operations. At n = 2048 that is 17 billion of them, per multiply.

In [ ]:
def time_matmul(n, device, reps=10):
    """Average seconds per n x n matmul on `device`."""
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)

    # Warm-up. The first call on a GPU pays for context setup and
    # kernel selection; timing it would measure the wrong thing.
    for _ in range(3):
        _ = torch.matmul(a, b)
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()
    for _ in range(reps):
        _ = torch.matmul(a, b)
    if device == "cuda":
        # CUDA calls are ASYNCHRONOUS: without this the timer stops
        # when the work was queued, not when it finished, and you
        # measure a speedup of 10000x that is not real.
        torch.cuda.synchronize()
    return (time.time() - start) / reps


N = 2048
cpu_t = time_matmul(N, "cpu")
print(f"CPU: {cpu_t * 1000:8.2f} ms per multiply   ({2 * N**3 / cpu_t / 1e9:.0f} GFLOP/s)")

if HAS_GPU:
    gpu_t = time_matmul(N, "cuda")
    print(f"GPU: {gpu_t * 1000:8.2f} ms per multiply   ({2 * N**3 / gpu_t / 1e9:.0f} GFLOP/s)")
    print(f"\nspeedup: {cpu_t / gpu_t:.0f}x")
else:
    gpu_t = None
    print("GPU: n/a")

Two things to take from that number.

**`torch.cuda.synchronize()` is not optional.** CUDA calls return
immediately and the work happens later. Time a GPU without it and you
will measure how fast Python can queue work — often a "1000× speedup"
that evaporates the moment you read a result back.

**The gap is architectural, not incremental.** A CPU has a handful of
fast cores optimised for branchy, sequential code. A GPU has thousands
of slow ones and can only use them when the same operation applies to
a great many numbers at once. A matmul is exactly that. So is a batch
of forward passes.

---

## 2. Training the model from warm-up 1

A matmul in a vacuum flatters the GPU. A real training loop also moves
data, computes a loss, runs backward, and updates parameters — and the
Python that orchestrates it runs on the CPU either way.

So measure the thing you actually care about: the same MLP, the same
epochs, both devices.

In [ ]:
def make_mlp(n_features, hidden_size, n_layers):
    """The warm-up 1 network, parameterised."""
    layers, d = [], n_features
    for _ in range(n_layers):
        layers += [nn.Linear(d, hidden_size), nn.ReLU()]
        d = hidden_size
    layers += [nn.Linear(d, 1)]
    return nn.Sequential(*layers)


def train_benchmark(X, y, device, n_epochs=200, hidden_size=256,
                    n_layers=4, verbose=True):
    """Train once, return (seconds, final loss, n_parameters)."""
    torch.manual_seed(0)
    X_dev, y_dev = X.to(device), y.to(device)
    model = make_mlp(X.shape[1], hidden_size, n_layers).to(device)

    # Adam, not SGD: this MLP is deep enough that plain SGD at a usable
    # learning rate diverges to nan on this data.
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()
    n_params = sum(p.numel() for p in model.parameters())

    for _ in range(5):                      # warm-up, not timed
        loss = loss_fn(model(X_dev), y_dev)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    if device == "cuda":
        torch.cuda.synchronize()

    start = time.time()
    for _ in range(n_epochs):
        loss = loss_fn(model(X_dev), y_dev)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.time() - start

    if verbose:
        print(f"  {n_layers} hidden layers x {hidden_size} units, {n_params:,} parameters")
        print(f"  {n_epochs} epochs on {device}: {elapsed:.2f} s   final loss {loss.item():.4f}")
    return elapsed, loss.item(), n_params


# A dataset big enough that the arithmetic dominates the overhead.
torch.manual_seed(42)
X_big = torch.randn(5000, 20)
y_big = (X_big @ torch.randn(20, 1)) + 0.5 * torch.randn(5000, 1)

print("CPU")
cpu_train, _, n_params = train_benchmark(X_big, y_big, "cpu")

if HAS_GPU:
    print("\nGPU")
    gpu_train, _, _ = train_benchmark(X_big, y_big, "cuda")
    print(f"\nspeedup: {cpu_train / gpu_train:.1f}x on {n_params:,} parameters")
else:
    gpu_train = None

The speedup here is real but **smaller than the raw matmul's**, and
that is the point of running both. A training step is not one big
matmul; it is a few small ones plus Python, plus an optimizer update,
plus a loss. Amdahl's law does the rest: the part you did not
accelerate becomes the part that costs.

---

## 3. Where the GPU stops paying

This is the section the workshop version does not have, and it is the
one that matters for this course.

Moving a tensor to the GPU and back costs time. Below some model size
that cost is larger than the arithmetic it saves, and the GPU is
**slower than the laptop**. Find that point.

In [ ]:
sizes = [8, 32, 128, 512]
rows = []
for h in sizes:
    c, _, npar = train_benchmark(X_big, y_big, "cpu", n_epochs=100,
                                 hidden_size=h, n_layers=2, verbose=False)
    g = None
    if HAS_GPU:
        g, _, _ = train_benchmark(X_big, y_big, "cuda", n_epochs=100,
                                  hidden_size=h, n_layers=2, verbose=False)
    rows.append((h, npar, c, g))

print(f"{'hidden':>7} {'params':>10} {'CPU (s)':>9} {'GPU (s)':>9} {'speedup':>9}")
for h, npar, c, g in rows:
    speed = f"{c / g:.2f}x" if g else "n/a"
    gtxt = f"{g:.2f}" if g else "n/a"
    print(f"{h:>7} {npar:>10,} {c:>9.2f} {gtxt:>9} {speed:>9}")

if HAS_GPU:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([r[1] for r in rows], [r[2] / r[3] for r in rows], "o-", color="#2f6f9f")
    ax.axhline(1.0, ls="--", c="#c1553b", lw=1, label="GPU = CPU")
    ax.set_xscale("log"); ax.set_xlabel("parameters")
    ax.set_ylabel("speedup (CPU time / GPU time)")
    ax.set_title("Below the dashed line, the GPU is slower")
    ax.legend(); ax.grid(alpha=0.3)
    plt.show()

Read the speedup column top to bottom. At the small end it is likely
**below 1** — the GPU loses. Every step still has to launch kernels and
synchronize, and on a tiny model that fixed cost is the whole cost.

This generalises into a rule worth keeping:

> A GPU pays when the arithmetic per step is large. It does not pay
> for small models, small batches, or work that is really a Python
> loop wearing a tensor costume.

**And that is why Session 4 does not need one.** The California-housing
MLP is 8-64-64-1 — about 4,800 parameters, in the region where the
curve above sits near or below 1. It trains in roughly ten seconds on
a Colab CPU. Switching that run to a GPU would not measurably help.

---

## Exercise — find the crossover, then push past it

Change the numbers below and re-run.

1. Increase `hidden_size` (`512`, `1024`, `2048`) with everything else
   fixed. At what parameter count does the GPU first win?
2. Now hold the model fixed and grow `n_samples` (`5000`, `50000`,
   `200000`). Does batch size move the crossover the same way model
   size does?
3. Set `n_epochs=20` and then `n_epochs=1000` at a size where the GPU
   wins. Does the *speedup* change? Should it?

Write down the answer to 1 — it is the number that decides, for the
rest of the year, whether a run belongs on your laptop or not.

In [ ]:
n_samples = 5000       # <- try 50000
n_features = 20        # <- try 100
hidden_size = 256      # <- try 512, 1024, 2048
n_layers = 4           # <- try 6
n_epochs = 200

torch.manual_seed(42)
X_exp = torch.randn(n_samples, n_features)
y_exp = (X_exp @ torch.randn(n_features, 1)) + 0.5 * torch.randn(n_samples, 1)

print("CPU")
c, _, npar = train_benchmark(X_exp, y_exp, "cpu", n_epochs=n_epochs,
                             hidden_size=hidden_size, n_layers=n_layers)
if HAS_GPU:
    print("\nGPU")
    g, _, _ = train_benchmark(X_exp, y_exp, "cuda", n_epochs=n_epochs,
                              hidden_size=hidden_size, n_layers=n_layers)
    print(f"\n{npar:,} parameters -> speedup {c / g:.2f}x")

---

## What to take away

1. **A GPU is a throughput device, not a fast CPU.** It wins on large
   batched arithmetic and loses on everything else.
2. **Always `torch.cuda.synchronize()` before stopping a timer.**
   Otherwise you are timing the queue, not the work.
3. **Warm up before measuring.** The first call pays setup costs that
   the next thousand do not.
4. **Measure before you migrate.** You now have a script that answers
   "would a GPU help here?" for any model you write this year.
5. **You do not need a GPU for Session 4.** You measured that.

**Next:** the challenge notebook, *AIE S4 — California Housing*.